# code for 2D trajectory embedding using UMAP

### imports

In [ ]:
import os
import math
import pickle
from os.path import join as opj
import pycircstat
import numpy as np
import pandas as pd
import hypertools as hyp
import matplotlib as mpl
from umap import UMAP
from statsmodels.stats.multitest import multipletests as mt

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
cmap = plt.cm.Spectral

### set paths

In [ ]:
data_dir = '../../data/'
ep_data_dir = opj(data_dir, 'models', 'episodes')
rec_data_dir = opj(data_dir, 'models', 'recalls')
ep_embs_dir = opj(ep_data_dir, 'embeddings')
rec_embs_dir = opj(rec_data_dir, 'embeddings')
pickle_dir = opj(data_dir, 'pickles')
fig_dir = '../../paper/figures/embeddings'

In [ ]:
if not os.path.isdir(fig_dir):
    os.mkdir(fig_dir)

### load data

In [ ]:
# trajectory embeddings
embeddings = {rectype: {} for rectype in ['atlep1', 'delayed', 'atlep2', 'arrdev']}

for rectype in embeddings.keys():
    embs = {}
    ep = 'atlep1' if rectype == 'delayed' else rectype
    embs['episode'] = np.load(opj(ep_embs_dir, f'{ep}.npy'))
    embs['recalls'] = {}
    recall_embs = [f for f in os.listdir(opj(rec_embs_dir, rectype)) if f.endswith('npy')]
    for file in recall_embs:
        if file.startswith('avg'):
            embs['avg_recall'] = np.load(opj(rec_embs_dir, rectype, file))
        else:
            embs['recalls'][os.path.splitext(file)[0]] = np.load(opj(rec_embs_dir, rectype, file))
    embeddings[rectype] = embs
    
# episode-recall event mappings
event_mappings = {rectype: np.load(opj(rec_events_dir, rectype, 'event_mappings.npy'))
                 for rectype in embeddings.keys()}

# session 1/session 2 psiturk ID mappings
id_maps = pd.read_pickle(opj(pickle_dir, 'id_maps.p'))

### define functions & classes

In [ ]:
class Point:
    def __init__(self, coord=None):
        self.coord = np.array(coord)

In [ ]:
class LineSegment:  
    def __init__(self, p1=None, p2=None):
        if isinstance(p1, Point):
            self.p1 = p1
        else:
            self.p1 = Point(p1)
            
        if isinstance(p2, Point):
            self.p2 = p2
        else:
            self.p2 = Point(p2)
        
    def intersect(self, z):
        if isinstance(z, Circle):
            return _seg_intersect_circle(self, z)
        elif isinstance(z, Rectangle):
            return _seg_intersect_rect(self, z)
        
    def norm(self):
        diff = self.p2.coord-self.p1.coord
        return diff/np.linalg.norm(diff)
    
    def get_p1(self):
        return self.p1.coord
    
    def get_p2(self):
        return self.p2.coord
    
    def get_vec(self):
        return self.p2.coord-self.p1.coord
        
    def angle(self, ref=None):
        if ref==None:
            p1 = np.zeros_like(self.get_p1())
            p2 = np.zeros_like(self.get_p1())
            p2[0] = 1
            ref = LineSegment(p1, p2)
        v0 = ref.get_vec()
        v1 = self.get_vec()
        return np.arccos(v0.dot(v1)/(np.linalg.norm(v0)*np.linalg.norm(v1)))

In [ ]:
class Circle:
    def __init__(self, center=None, r=None):
        self.center = np.array(center)
        self.r = r 
    
    def get_center(self):
        return self.center
    
    def get_radius(self):
        return self.r

In [ ]:
class Rectangle:
    def __init__(self, x=None, y=None, w=None):
        self.c0 = x-w
        self.c1 = y-w
        self.c2 = x+w
        self.c3 = y+w

In [ ]:
def _seg_intersect_circle(ls, circ):
     
    Q = circ.get_center()
    r = circ.get_radius()
    P1 = ls.get_p1()
    V = ls.get_p2() - P1
    
    a = V.dot(V)
    b = 2 * V.dot(P1 - Q)
    c = P1.dot(P1) + Q.dot(Q) - 2 * P1.dot(Q) - r**2
    
    disc = b**2 - 4 * a * c
    if disc < 0:
        return False
    
    sqrt_disc = math.sqrt(disc)
    t1 = (-b + sqrt_disc) / (2 * a)
    t2 = (-b - sqrt_disc) / (2 * a)
    if not (0 <= t1 <= 1 or 0 <= t2 <= 1):
        return False
    
    return True

In [ ]:
def _seg_intersect_rect(ls, r):
    
    # find min/max X for the segment
    minX = min(ls.p1.x, ls.p2.x)
    maxX = max(ls.p1.x, ls.p2.x)
    
    # find the intersection of the segment's and rectangle's x-projections
    if maxX > r.c2:
        maxX = r.c2
    if minX < r.c0:
        minX = r.c0
    
    if minX > maxX:
        return False
    
    minY = ls.p1.y
    maxY = ls.p2.y
    
    dx = ls.p2.x - ls.p1.x
    
    if abs(dx) > .0000001:
        a = (ls.p2.y - ls.p1.y) / dx
        b = ls.p1.y - a * ls.p1.x
        minY = a * minX + b
        maxY = a * maxX + b
        
    if minY > maxY:
        tmp = maxY
        maxY = minY
        minY = tmp
        
    # find the intersection of the segment's and rectangle's y-projections
    if maxY > r.c3:
        maxY = r.c3
    if minY < r.c1:
        minY = r.c1
        
    # if Y-projections do not intersect return false
    if minY > maxY:
        return False
    else:
        return True

In [ ]:
def add_arrows(axes, x, y, minifig=False, **kwargs):

    if minifig:
        aspace = 0.1
    else:
        aspace = .05
        
    aspace *= scale

    r = [0]
    for i in range(1,len(x)):
        dx = x[i]-x[i-1]
        dy = y[i]-y[i-1]
        r.append(np.sqrt(dx*dx+dy*dy))
    r = np.array(r)

    rtot = []
    for i in range(len(r)):
        rtot.append(r[0:i].sum())
    rtot.append(r.sum())

    arrowData = []
    arrowPos = 0
    rcount = 1 
    while arrowPos < r.sum():
        x1,x2 = x[rcount-1],x[rcount]
        y1,y2 = y[rcount-1],y[rcount]
        da = arrowPos-rtot[rcount] 
        theta = np.arctan2((x2-x1),(y2-y1))
        ax = np.sin(theta)*da+x1
        ay = np.cos(theta)*da+y1
        arrowData.append((ax,ay,theta))
        arrowPos+=aspace
        while arrowPos > rtot[rcount+1]: 
            rcount+=1
            if arrowPos > rtot[-1]:
                break

    for ax,ay,theta in arrowData:
        axes.arrow(ax,ay,
                   np.sin(theta)*aspace/10,np.cos(theta)*aspace/10, 
                   head_width=aspace/3, **kwargs)

In [ ]:
def _compute_coord(xi, yi, w, seglist, kind='rectangle'):
    
    if kind=='rectangle':
        z = Rectangle(x=xi, y=yi, w=w)
    elif kind=='circle':
        z = Circle(center=[xi,yi], r=w)
        
    segs = list(filter(lambda s: s.intersect(z), seglist))
    
    if len(segs)>1:
        u, v  = np.array([seg.norm() for seg in segs]).mean(0)
        rads = np.array([seg.angle() for seg in segs])
        p, z = pycircstat.tests.rayleigh(rads)
    else:
        u = 0
        v = 0
        p = 1
    c = len(segs)
    return u, v, p, c

In [ ]:
def _compute_grid_stats(episode, recall_dict):
    # create 2D grid
    scale = np.abs(episode).max()
    step = scale / 25
    X, Y = np.meshgrid(np.arange(-scale, scale, step), np.arange(-scale, scale, step))

    # turn embedded recall event model into a list of line segments
    seglist = []
    for i, (turkid, sub_emb) in enumerate(recall_dict.items()):
        for j in range(sub_emb.shape[0] - 1):
            p1 = Point(coord=sub_emb[j, :])
            p2 = Point(coord=sub_emb[j + 1, :])
            seg = LineSegment(p1=p1, p2=p2)

            seglist.append(seg)

    # compute the average vector and p-value at each grid point
    U = np.zeros_like(X)
    V = np.zeros_like(X)
    P = np.zeros_like(X)
    # Z = np.zeros_like(X)
    C = np.zeros_like(X)
    for i, (x, y) in enumerate(zip(X, Y)):
        for j, (xi, yi) in enumerate(zip(x, y)):
            U[i, j], V[i, j], P[i, j], C[i, j] = _compute_coord(xi, yi, step * 2, seglist, kind='circle')

    # multiple comparisons correction
    thresh = .001
    Pc = mt(P.ravel(), method='fdr_bh', alpha=.05)[1].reshape(np.shape(X))
    M = np.hypot(U, V)
    M = plt.cm.Blues(M)
    M[Pc > thresh] = [.5, .5, .5, .1]
    M[Pc == 1] = [.5, .5, .5, 0]
    
    return X, Y, U, V, M, scale

In [ ]:
def plot_embeddings(embeddings_dict, save_path=None, suptitle=None):
    episode = embeddings_dict['episode']
    avg_recall = embeddings_dict['avg_recall']
    recalls = embeddings_dict['recalls']
    mappings = embeddings_dict['mapping']
    
    # compute average recall statistics
    X, Y, U, V, M, scale = _compute_grid_stats(episode, recalls)
    
    # create figure with subplots
    plt.figure(figsize=(12, (len(recalls) // 8) * 2))
    mpl.rcParams['pdf.fonttype'] = 42
    axarr = [0 for i in range(2)]
    
    axarr[0] = plt.subplot2grid((len(recalls) // 8 + 3, 8), (0, 1), colspan=3, rowspan=2)
    axarr[1] = plt.subplot2grid((len(recalls) // 8 + 3, 8), (0, 4), colspan=3, rowspan=2)
    
    for i in range(2, (len(recalls) // 8 + 3)):
        for j in range(0, 8):
            ax = plt.subplot2grid((len(recalls) // 8 + 3, 8), (i, j))
            axarr.append(ax)
   
    # plot episode trajectory and events
    axarr[0].scatter(episode[:, 0], episode[:, 1], c=range(episode.shape[0]),
                     cmap=cmap, s=150, zorder=3)
    axarr[0].scatter(episode[:, 0], episode[:, 1], c='k', cmap=cmap, s=200, zorder=2)
    axarr[0].plot(episode[:, 0], episode[:, 1], zorder=1, c='k', alpha=.5)
    add_arrows(axarr[0], episode[:, 0], episode[:, 1], scale, zorder=0, alpha=1, color='k', fill=True)
    axarr[0].set_title('Episode events')
    axarr[0].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
    axarr[0].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
    axarr[0].text(0, 1, 'A', horizontalalignment='center', transform=axarr[0].transAxes, fontsize=18)
    
    # plot average recall events
    axarr[1].quiver(X, Y, U, V, color=M.reshape(M.shape[0] * M.shape[1], 4), zorder=1, width=.004)
    axarr[1].plot(avg_recall[:, 0], avg_recall[:, 1], zorder=2, c='k', alpha=1)
    axarr[1].plot(episode[:, 0], episode[:, 1], zorder=1, c='k', alpha=.5)
    add_arrows(axarr[1], avg_recall[:, 0], avg_recall[:, 1], scale, add_arrows, zorder=3, alpha=1, color='k', fill=True)
    axarr[1].scatter(avg_recall[:, 0], avg_recall[:, 1], c=range(avg_recall.shape[0]), cmap=cmap,
                     s=150, zorder=4)
    axarr[1].scatter(avg_recall[:, 0], avg_recall[:, 1], c='k', cmap=cmap, s=200, zorder=3)
    axarr[1].set_title('Average recall events')
    axarr[1].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
    axarr[1].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
    axarr[1].text(0, 1, 'B',
                  horizontalalignment='center',
                  transform=axarr[1].transAxes,
                  fontsize=18)

    # plot individual recalls
    axarr[2].text(0, 1.05, 'C', horizontalalignment='center', transform=axarr[2].transAxes, fontsize=18)

    if rectype == 'atlep1':
        ids = id_maps['session 1']
    elif rectype == 'delayed':
        ids = id_maps['session 2']
    elif rectype == 'atlep2':
        ids = id_maps.loc[id_maps.index.str.contains('A'), 'session 2']
    else:
        ids = id_maps.loc[id_maps.index.str.contains('B'), 'session 2']

    for i, turkid in enumerate(ids, start=2):
        rec_emb = recalls[turkid]
        m = mappings[np.where(mappings.T[0] == turkid)].ravel()[1]
        axarr[i].scatter(rec_emb[:, 0], rec_emb[:, 1], c=cmap(m / len(episode)), cmap=cmap, s=60, zorder=2)
        axarr[i].plot(rec_emb[:, 0], rec_emb[:, 1], zorder=1, c='k', alpha=.25)
        axarr[i].plot(avg_recall[:, 0], avg_recall[:, 1], zorder=3, c='k', alpha=1)
        add_arrows(axarr[i], rec_emb[:, 0], rec_emb[:, 1], scale, zorder=1, alpha=.25, color='k', minifig=True, fill=True)
        axarr[i].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
        axarr[i].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
        axarr[i].set_title(f'P{i - 1}')

    for a in axarr:
        a.axis('off')

    if suptitle:
        plt.suptitle(suptitle, y=1.05, fontsize=20)
    plt.tight_layout()
    plt.subplots_adjust(wspace=0, hspace=0.25)
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

# plot trajectory embeddings

In [ ]:
for rectype, embs in embeddings.items():
    save_path = opj(fig_dir, f'{rectype}_embedding.pdf')
    plot_embedding(embds, save_path=save_path)

## old code for immediate & delayed on same axes

In [ ]:
# # shorten some variable names
# episode_es = imm_del_embs['episode_events']
# avg_p_imm = imm_del_embs['avg_participant_immediate']
# avg_p_del = imm_del_embs['avg_participant_delayed']
# ps_imm = imm_del_embs['participants_immediate']
# ps_del = imm_del_embs['participants_delayed']

# imm_mappings = event_mappings['atlep1']
# del_mappings = event_mappings['delayed']

# # # create a 2D grid

# # # scale = int(np.argmax([abs(i) for i in episode_es]))
# # scale = int(np.concatenate([abs(i[1]) for i in ps_imm]+[abs(i[1]) for i in ps_del]).max())+1
# scale = np.abs(episode_es).max()+1
# step = scale / 24

# X, Y = np.meshgrid(np.arange(-scale, scale, step), np.arange(-scale, scale, step))

# # turn embedded recall event model into a list of line segments
# imm_seglist = []
# for i, (turkid, sub_emb) in enumerate(ps_imm):
#     for j in range(sub_emb.shape[0]-1):
#         p1 = Point(coord=sub_emb[j, :])
#         p2 = Point(coord=sub_emb[j+1, :])
#         seg = LineSegment(p1=p1, p2=p2)

#         imm_seglist.append(seg)
        
# del_seglist = []
# for i, (turkid, sub_emb) in enumerate(ps_del):
#     for j in range(sub_emb.shape[0]-1):
#         p1 = Point(coord=sub_emb[j, :])
#         p2 = Point(coord=sub_emb[j+1, :])
#         seg = LineSegment(p1=p1, p2=p2)

#         del_seglist.append(seg)

# # compute the average vector and p-value at each grid point
# iU = np.zeros_like(X)
# iV = np.zeros_like(X)
# iP = np.zeros_like(X)
# iZ = np.zeros_like(X)
# iC = np.zeros_like(X)
# for i, (x, y) in enumerate(zip(X, Y)):
#     for j, (xi, yi) in enumerate(zip(x, y)):
#         iU[i, j], iV[i, j], iP[i, j], iC[i, j] = compute_coord(xi, yi, step*2, imm_seglist, kind='circle')

# dU = np.zeros_like(X)
# dV = np.zeros_like(X)
# dP = np.zeros_like(X)
# dZ = np.zeros_like(X)
# dC = np.zeros_like(X)
# for i, (x, y) in enumerate(zip(X, Y)):
#     for j, (xi, yi) in enumerate(zip(x, y)):
#         dU[i, j], dV[i, j], dP[i, j], dC[i, j] = compute_coord(xi, yi, step*2, del_seglist, kind='circle')

# # multiple comparisons correction
# thresh = .001
# iPc = mt(iP.ravel(), method='fdr_bh', alpha=.05)[1].reshape(48,48)
# iM = np.hypot(iU, iV)
# iM = plt.cm.Blues(iM)
# iM[iPc>thresh]=[.5, .5, .5, .25]
# iM[iPc==1]=[.5, .5, .5, 0]

# thresh = .001
# dPc = mt(dP.ravel(), method='fdr_bh', alpha=.05)[1].reshape(48,48)
# dM = np.hypot(dU, dV)
# dM = plt.cm.Blues(dM)
# dM[dPc>thresh]=[.5, .5, .5, .25]
# dM[dPc==1]=[.5, .5, .5, 0]


# # create figure with subplots
# plt.figure(figsize=(16,26))
# mpl.rcParams['pdf.fonttype'] = 42
# axarr = [0 for i in range(3)]

# # axarr[0] = plt.subplot2grid((8, 6), (0, 0), colspan=6, rowspan=4)
# # axarr[1] = plt.subplot2grid((8, 6), (4, 0), colspan=3, rowspan=2)
# # axarr[2] = plt.subplot2grid((8, 6), (4, 3), colspan=3, rowspan=2)

# # axarr[0] = plt.subplot2grid((8, 10), (1, 0), rowspan=6, colspan=6)
# # axarr[1] = plt.subplot2grid((8, 10), (0, 6), rowspan=4, colspan=4)
# # axarr[2] = plt.subplot2grid((8, 10), (4, 6), rowspan=4, colspan=4)

# axarr[0] = plt.subplot2grid((28, 16), (0, 0), rowspan=8, colspan=9)
# axarr[1] = plt.subplot2grid((28, 16), (0, 9), rowspan=4, colspan=5)
# axarr[2] = plt.subplot2grid((28, 16), (4, 9), rowspan=4, colspan=5)

# for i in range(8, 24, 2):
#     for j in range(0, 16, 2):
#         ax = plt.subplot2grid((28,16), (i, j), rowspan=2, colspan=2)
#         axarr.append(ax)


# # plot episode trajectory and events
# axarr[0].scatter(episode_es[:, 0], episode_es[:, 1], c=range(episode_es.shape[0]), 
#                  cmap=cmap, s=300, zorder=3)
# axarr[0].scatter(episode_es[:, 0], episode_es[:, 1], c='k', cmap=cmap, s=400, zorder=2)
# axarr[0].plot(episode_es[:, 0], episode_es[:, 1], zorder=1, c='k', alpha=.5)
# #axarr[0].plot(episode_trajectory[:, 0], episode_trajectory[:, 1], zorder=1, c='k', alpha=.5)
# add_arrows(axarr[0], episode_es[:, 0], episode_es[:, 1], zorder=0, alpha=1, color='k', fill=True, label='Episode')
# axarr[0].plot(avg_p_imm[:, 0], avg_p_imm[:, 1], zorder=2, c='k', alpha=0.8, label='Immediate')
# axarr[0].plot(avg_p_del[:, 0], avg_p_del[:, 1], zorder=2, c='k', alpha=0.8, linestyle='--', label='Delayed')
# axarr[0].set_title('Episode events')
# axarr[0].set_xlim(-scale, scale)
# axarr[0].set_ylim(-scale, scale)
# axarr[0].legend(loc=2, fontsize='xx-large')
# axarr[0].text(0, 1,'A',
#         horizontalalignment='center',
#         transform=axarr[0].transAxes,
#           fontsize=18)

# # # plot average immediate recall events
# axarr[1].quiver(X, Y, iU, iV, color=iM.reshape(iM.shape[0]*iM.shape[1],4), zorder=1, width=.004)
# axarr[1].plot(avg_p_imm[:, 0], avg_p_imm[:, 1], zorder=2, c='k', alpha=1)
# axarr[1].plot(episode_es[:, 0], episode_es[:, 1], zorder=1, c='k', alpha=.5)
# # axarr[1].plot(avg_p_del[:, 0], avg_p_del[:, 1], zorder=2, c='k', alpha=0.8, linestyle='--')
# add_arrows(axarr[1], avg_p_imm[:, 0], avg_p_imm[:, 1], zorder=3, alpha=1, color='k', fill=True)
# axarr[1].scatter(avg_p_imm[:, 0], avg_p_imm[:, 1], c=range(avg_p_imm.shape[0]), cmap=cmap, 
#                  s=150, zorder=4)
# #axarr[1].plot(avg_p_trajectory[:, 0], avg_p_trajectory[:, 1], zorder=1, c='k', alpha=.5)
# axarr[1].scatter(avg_p_imm[:, 0], avg_p_imm[:, 1], c='k', cmap=cmap, s=200, zorder=3)
# axarr[1].set_title('Average immediate recall')
# axarr[1].set_xlim(-scale, scale)
# axarr[1].set_ylim(-scale, scale)
# axarr[1].text(0, 1,'B',
#         horizontalalignment='center',
#         transform=axarr[1].transAxes,
#           fontsize=18)

# # # plot average delayed recall events
# axarr[2].quiver(X, Y, dU, dV, color=dM.reshape(dM.shape[0]*dM.shape[1],4), zorder=1, width=.004)
# # axarr[2].plot(avg_p_imm[:, 0], avg_p_imm[:, 1], zorder=2, c='k', alpha=0.8)
# axarr[2].plot(avg_p_del[:, 0], avg_p_del[:, 1], zorder=2, c='k', alpha=1)
# axarr[2].plot(episode_es[:, 0], episode_es[:, 1], zorder=1, c='k', alpha=.5)
# add_arrows(axarr[2], avg_p_del[:, 0], avg_p_del[:, 1], zorder=3, alpha=1, color='k', fill=True)
# axarr[2].scatter(avg_p_del[:, 0], avg_p_del[:, 1], c=range(avg_p_del.shape[0]), cmap=cmap, 
#                  s=150, zorder=4)
# #axarr[1].plot(avg_p_trajectory[:, 0], avg_p_trajectory[:, 1], zorder=1, c='k', alpha=.5)
# axarr[2].scatter(avg_p_del[:, 0], avg_p_del[:, 1], c='k', cmap=cmap, s=200, zorder=3)
# axarr[2].set_title('Average delayed recall')
# axarr[2].set_xlim(-scale, scale)
# axarr[2].set_ylim(-scale, scale)
# axarr[2].text(0, 1,'C',
#         horizontalalignment='center',
#         transform=axarr[2].transAxes,
#           fontsize=18)

# # # plot individual participants
# axarr[3].text(-1.75, 0, 'D', horizontalalignment='left', transform=axarr[2].transAxes, fontsize=18)

# for i, (turkid1, ie) in enumerate(indv_imm):
#     sid, turkid2 = next((sid, id_maps[sid]['session 2']) for sid in id_maps.keys() 
#                         if id_maps[sid]['session 1'] == turkid1)
#     de = next(e for tid2, e in indv_del if tid2==turkid2)
#     dm = next(m for tid2, m in del_mappings if tid2==turkid2)
#     for tid1, im in imm_mappings:
#         for t1, _ in participant_trajectories['atlep1']:
#             if turkid1 == tid1 == t1:
#                 axarr[i+3].scatter(ie[:,0], ie[:,1], c=cmap(im/34), cmap=cmap, s=60, zorder=3, alpha=0.75)
#     #             axarr[i+3].scatter(ie[:,0], ie[:,1], c='red', cmap=cmap, s=100, zorder=2, alpha=0.75)
#                 axarr[i+3].scatter(ie[:, 0], ie[:, 1], s=100, zorder=3, facecolors='none', edgecolors='k')
# #                 add_arrows(axarr[i+3], ie[:, 0], ie[:, 1], zorder=1, alpha=1, color='k', minifig=True, fill=True)
#                 axarr[i+3].scatter(de[:,0], de[:,1], c=cmap(dm/34), cmap=cmap, s=100, zorder=3, alpha=0.75)
# #                 add_arrows(axarr[i+3], de[:, 0], de[:, 1], zorder=1, alpha=.5, color='k', minifig=True, fill=True)
#     #             axarr[i+3].scatter(de[:,0], de[:,1], c='blue', cmap=cmap, s=100, zorder=2, alpha=0.75)
#     #             axarr[i+3].plot(ie[:,0], ie[:,1], zorder=1, c='k', alpha=.8)
#     #             axarr[i+3].plot(de[:,0], de[:,1], zorder=1, c='k', alpha=.3, linestyle='--')
#     #             axarr[i+3].plot(avg_p_imm[:, 0], avg_p_imm[:, 1], zorder=1, c='k', alpha=1)
#                 axarr[i+3].plot(episode_es[:, 0], episode_es[:, 1], zorder=1, c='k', alpha=.5)
#     #             add_arrows(axarr[i+3], ie[:,0], ie[:,1], zorder=1, alpha=.25, color='k', fill=True)
#     #             add_arrows(axarr[i+3], de[:,0], de[:,1], zorder=1, alpha=.25, color='k', fill=True)
#     #             #axarr[i+3].plot(episode_trajectory[:, 0], episode_trajectory[:, 1], c='k', zorder=3)
#     #             #axarr[i+3].plot(episode_es[:, 0], episode_es[:, 1], c='k', zorder=3)
#                 axarr[i+3].set_xlim(-scale, scale)
#                 axarr[i+3].set_ylim(-scale, scale)
#                 axarr[i+3].set_title(f'P{i+1}')

# for a in axarr:
#     a.axis('off')

# plt.tight_layout()
# plt.subplots_adjust(wspace=.05, hspace=.25)
# # plt.suptitle('Immediate vs Delayed Recall', y=1.025, ha='center', fontsize=20)
# # plt.savefig('../../figures/embeddings/imm-del-embedding.pdf')
# # plt.savefig('/Users/paxtonfitzpatrick/Desktop/imm-del-embedding.pdf')